In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np

In [ ]:
# read dataset with all features
data = pd.read_csv("../data/curated/merged_data.csv")
data["city_centre"] = data["city_centre"].astype(bool)
data["coastal"] = data["coastal"].astype(bool)
print(list(data.columns))

In [ ]:
# remove properites with over 2000 weekly rent 
print(len(data))
data = data[data["weekly rent amount"] <= 2000]
print(len(data))

# Predict 2023 values of features that are likely to change from time of data collection to 2023

In [ ]:
def predict_2023_value(y_values, col_name, x_values = [0,5,10], predict_val = 12): 
    """ 
    Function predicts 2023 value of variable given past data. Assumes linear relationship.
    default value of x_value 0, 5, 10 represents 2011, 2016, 2021. 
    default value of predict_val represents 2023
    """
    # 0, 5, 10 represents 2011, 2016, 2021
    print(y_values)
    print(col_name)
    model = LinearRegression()
    x_values = np.array(x_values).reshape(-1,1)
    y_values = np.array(y_values).reshape(-1,1)
    model.fit(x_values, y_values)
    # 12 represents 2023
    predict_2023 = model.predict(np.array(predict_val).reshape(-1,1))
    return pd.Series({col_name: predict_2023[0][0]})

In [ ]:
# predict median equivalised total household income (weekly) ($) 2023
median_income_2023_column = data.apply(lambda x:
    predict_2023_value([x["median equivalised total household income (weekly) ($) 2011"], 
     x["median equivalised total household income (weekly) ($) 2016"],
     x["median equivalised total household income (weekly) ($) 2021"]], 
    "predicted median equivalised total household income (weekly) ($) 2023"), axis=1)

data = pd.concat([data, median_income_2023_column], axis=1)

data = data.drop(["median equivalised total household income (weekly) ($) 2011", 
     "median equivalised total household income (weekly) ($) 2016",
     "median equivalised total household income (weekly) ($) 2021"], axis=1)

In [ ]:
# median weekly household rental payment ($) 2023
weekly_rent_2023_column = data.apply(lambda x:
    predict_2023_value([x["median weekly household rental payment ($) 2011"], 
     x["median weekly household rental payment ($) 2016"],
     x["median weekly household rental payment ($) 2021"]], 
    "predicted median weekly household rental payment ($) 2023"), axis=1)

data = pd.concat([data, weekly_rent_2023_column], axis=1)


data = data.drop(["median weekly household rental payment ($) 2011", 
     "median weekly household rental payment ($) 2016",
     "median weekly household rental payment ($) 2021"], axis=1)



In [ ]:
# predict average family size (no. of persons) 2023
weekly_rent_2023_column = data.apply(lambda x:
    predict_2023_value([x["average family size (no. of persons) 2011"], 
     x["average family size (no. of persons) 2016"],
     x["average family size (no. of persons) 2021"]], 
    "predicted average family size (no. of persons) 2023"), axis=1)

data = pd.concat([data, weekly_rent_2023_column], axis=1)

data = data.drop(["average family size (no. of persons) 2011", 
     "average family size (no. of persons) 2016",
     "average family size (no. of persons) 2021"], axis=1)


In [ ]:
# predict median weekly household rental payment ($) 2023
weekly_rent_2023_column = data.apply(lambda x:
    predict_2023_value([x["employment ratio 2011"], 
     x["employment ratio 2016"],
     x["employment ratio 2021"]], 
    "predicted employment ratio 2023"), axis=1)

data = pd.concat([data, weekly_rent_2023_column], axis=1)

data = data.drop(["employment ratio 2011", 
     "employment ratio 2016",
     "employment ratio 2021"], axis=1)

In [ ]:
# predict estimated resident population (no.) 2023
weekly_rent_2023_column = data.apply(lambda x:
    predict_2023_value([x["estimated resident population (no.) 2011"], 
     x["estimated resident population (no.) 2016"],
     x["estimated resident population (no.) 2021"]], 
    "predicted estimated resident population (no.) 2023"), axis=1)

data = pd.concat([data, weekly_rent_2023_column], axis=1)

data = data.drop(["estimated resident population (no.) 2011", 
     "estimated resident population (no.) 2016",
     "estimated resident population (no.) 2021"], axis=1)

In [ ]:
# predict total dwellings (no.) 2023
dwelling_2023_column = data.apply(lambda x:
    predict_2023_value([x["total dwellings (no.) 2017"], 
     x["total dwellings (no.) 2019"],
     x["total dwellings (no.) 2021"]], 
    "predicted total dwellings (no.) 2023", [0,2,4], 6), axis=1)

data = pd.concat([data, dwelling_2023_column], axis=1)

data = data.drop(["total dwellings (no.) 2017", 
     "total dwellings (no.) 2019",
     "total dwellings (no.) 2021"], axis=1)

In [ ]:
print(list(data.columns))

In [ ]:
data.isna().any()

In [ ]:
# drop columns with a lot of missing values 
data = data.drop(["total_supermarkets", "major_supermarkets"], axis=1)
list(data.columns[data.isna().any()])

In [ ]:
# drop remaining rows with missing values
print(len(data))
data = data.dropna()
print(len(data))

In [ ]:
# save data for xgboost before one-hot-encoding with postcode which is not used in XGBoost
data.to_csv("../data/curated/data_XGBoost_before_onehotencode_with_postcode.csv", index=False)
data = data.drop(["postcode"], axis=1)
# save data for xgboost before one-hot-encoding which is used in XGBoost
data.to_csv("../data/curated/data_XGBoost_before_onehotencode.csv", index=False)

In [ ]:
# one hot encode categorical variables
data = pd.get_dummies(data, columns=["property type", "closest_city_centre"])
data

# Correlation between continous features

In [ ]:
continous_data = data.select_dtypes(include=['number'])
continous_data

In [ ]:
sns.heatmap(continous_data.corr())

plt.title('Pearson Correlation Metric')
plt.show()

# XGBoost Model

In [ ]:
# split to training and test set
y = data["weekly rent amount"]
X = data.drop(["weekly rent amount"], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error



# Perform grid search to tune hyperparameters
param_grid = {
    'n_estimators': [50, 70, 100],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
}

xgb_regressor = XGBRegressor()


grid_search = GridSearchCV(estimator=xgb_regressor, param_grid=param_grid, 
                           scoring='neg_mean_squared_error', cv=5)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train)

# Get the best hyperparameters from the grid search
best_params = grid_search.best_params_
print("Best Hyperparameters:", best_params)

# Get the best estimator (model)
best_xgb_model = grid_search.best_estimator_

# Evaluate the model on the test set
predictions = best_xgb_model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
print("Mean Squared Error on Test Set:", mse)
rmse = np.sqrt(mse)
print("Root Mean Squared Error (RMSE):", rmse)


In [ ]:
# create dataframe with true value, prediction and absolute error
pred_true_val = pd.DataFrame({"true rent price": y_test, "predicted price": predictions})
pred_true_val["error"] = np.abs(pred_true_val["true rent price"] - pred_true_val["predicted price"])

# Most important features

In [ ]:
# get of number of times a feature is used to split in tree across all trees 
booster = best_xgb_model.get_booster()
feature_scores = booster.get_score(importance_type='weight')
feature_scores_df = pd.DataFrame({"feature": list(feature_scores.keys()), "weight": list(feature_scores.values())})
feature_scores_df = feature_scores_df.sort_values(by=["weight"], ascending=False)
feature_scores_df.to_csv("../data/curated/top_features.csv", index=False)
feature_scores_df


# Plots of model perfromance

In [ ]:
# distribution true rent price
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(pred_true_val["true rent price"])
ax.set_title("Histogram of true rent price")

In [ ]:
# distribution predicted price
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(pred_true_val["predicted price"])
ax.set_title("Histogram of predicted rent price")

In [ ]:
# Histogram of errors
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(pred_true_val["error"])
ax.set_xlabel("Error value")
ax.set_ylabel("Count")
ax.set_title("Histogram of Errors")

In [ ]:
# Boxplot of errors
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(pred_true_val["error"])
ax.set_xlabel("Error value")
ax.set_ylabel("Count")
ax.set_title("Boxplot of Errors")
print(f"median error {np.median(pred_true_val['error'])}")
print(f"25% percentile error {np.percentile(pred_true_val['error'], 25)}")
print(f"75% percentile error {np.percentile(pred_true_val['error'], 75)}")

In [ ]:
# average error for each rent price 
average_error_rent_price = pred_true_val.groupby("true rent price")["error"].mean().reset_index()
average_error_rent_price


In [ ]:
# Bar plot of average errors for rental price
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x="true rent price", y="error", data=average_error_rent_price, ax=ax)
tick_interval = 20  # Show every 20th label
x_values = list(average_error_rent_price["true rent price"])
visible_indices = range(0, len(x_values), tick_interval) 
visible_ticks = [x_values[i] for i in visible_indices]
ax.set_xticks(visible_indices)
ax.set_xticklabels(visible_ticks, rotation=45, ha='right')  # Rotate and align the labels
ax.set_xlabel("True Rent Price")
ax.set_ylabel("Error Prediction")
ax.set_title("True Rent Price vs Averate Prediction Error Bar Plot")


In [48]:
import requests
ORS_BASE_URL = "http://localhost:8080/ors/v2/directions/driving-car"
first_point = ["144.970354","-37.7786286"]
second_point = ["145.1094256257614","-37.79812686713807"]
url = f'{ORS_BASE_URL}?start={first_point[0]},{first_point[1]}&end={second_point[0]},{second_point[1]}'
print(url)
response = requests.get(url)

print(response.reason)

json_response = response.json()

http://localhost:8080/ors/v2/directions/driving-car?start=144.970354,-37.7786286&end=145.1094256257614,-37.79812686713807



In [49]:
print(response.text)

{"error":{"code":2099,"message":"Unable to get an appropriate route profile for RoutePreference = driving-car"},"info":{"engine":{"build_date":"2023-09-20T12:53:44Z","version":"8.0"},"timestamp":1697455924061}}


In [50]:
json_response

{'error': {'code': 2099,
  'message': 'Unable to get an appropriate route profile for RoutePreference = driving-car'},
 'info': {'engine': {'build_date': '2023-09-20T12:53:44Z', 'version': '8.0'},
  'timestamp': 1697455924061}}